## Import libraries

In [4]:
import os
import requests
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torchvision
from torchvision import models
from pinecone import Pinecone, ServerlessSpec
from dotenv import load_dotenv, dotenv_values 

load_dotenv()

INDEX_DIMENSION=1024 # defined based on convnext "base" model setting for embeggings
PINECONE_KEY=os.getenv("PINECONE_KEY")
INDEX_NAME=os.getenv("INDEX_NAME")


## EDA (tbd)

Load .csv, .json and images data

In [9]:
styles_metadata = pd.read_csv("../fashion-dataset/styles.csv", delimiter=';')
images_metadata = pd.read_csv("../fashion-dataset/images.csv", delimiter=';')

In [10]:
image_ids = np.sort(styles_metadata.id)
image_address = ["../fashion-dataset/images/"+str(i)+".jpg" for i in image_ids]
image_address_dictionary = {k:v for k,v in zip(image_ids,image_address )}

In [11]:
image_set = {}
missed_items_ids = []
for id in image_address_dictionary:
    address = image_address_dictionary[id]
    try:
        image_set[id] = Image.open(address)
    except:
        missed_items_ids.append(id)

In [12]:
missed_items_ids

[np.int64(12347),
 np.int64(39401),
 np.int64(39403),
 np.int64(39410),
 np.int64(39425)]

# 1. Create Pinecone Index

In [4]:
# initialize Pinecone
pc = Pinecone(api_key=PINECONE_KEY)

In [5]:
# create index
if INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=INDEX_NAME, 
        dimension=INDEX_DIMENSION,
        spec=ServerlessSpec(
            cloud='aws',
            region='us-east-1'
        )
    )

index = pc.Index(INDEX_NAME)


In [6]:
index

# 2. Prepare embeddings

In [5]:
# loading pretrained ConvNeXT
# URL for ConvNeXt Base weights
url = "https://download.pytorch.org/models/convnext_base-6075fbad.pth"

# download the weights
response = requests.get(url)
with open("convnext_base.pth", "wb") as f:
    f.write(response.content)

# load the model with the downloaded weights
model = models.convnext_base()
model.load_state_dict(torch.load("convnext_base.pth"))


/var/folders/73/8x9p3dg55zz90b9r5c3w9nlh0000gn/T/ipykernel_52177/3973720482.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("convnext_b

<All keys matched successfully>

In [6]:
model.eval()

ConvNeXt(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 128, kernel_size=(4, 4), stride=(4, 4))
      (1): LayerNorm2d((128,), eps=1e-06, elementwise_affine=True)
    )
    (1): Sequential(
      (0): CNBlock(
        (block): Sequential(
          (0): Conv2d(128, 128, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=128)
          (1): Permute()
          (2): LayerNorm((128,), eps=1e-06, elementwise_affine=True)
          (3): Linear(in_features=128, out_features=512, bias=True)
          (4): GELU(approximate='none')
          (5): Linear(in_features=512, out_features=128, bias=True)
          (6): Permute()
        )
        (stochastic_depth): StochasticDepth(p=0.0, mode=row)
      )
      (1): CNBlock(
        (block): Sequential(
          (0): Conv2d(128, 128, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=128)
          (1): Permute()
          (2): LayerNorm((128,), eps=1e-06, elementwise_affine=True)
          (3): Linear(

In [8]:
model.classifier

Sequential(
  (0): LayerNorm2d((1024,), eps=1e-06, elementwise_affine=True)
  (1): Flatten(start_dim=1, end_dim=-1)
  (2): Linear(in_features=1024, out_features=1000, bias=True)
)